# SAMSum Integrated Demonstration

This notebook demonstrates one end-to-end training-split example: load a dialogue, generate a structured result (summary, key points, decisions, action items), validate it, invoke the allowlisted validation function, then generate a final response conditioned on the tool result. The structured result is shown in a readable table for review; the complete machine-readable JSON remains saved by the CLI. Generated text must be reviewed against the dialogue; schema validity does not establish factual correctness.

Run from the repository root after setting up the environment and obtaining/preprocessing the dataset. The first model run may download Qwen weights and can take several minutes on CPU.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
PROJECT_MARKER = Path("src") / "integrated_demo" / "run_integrated_demo.py"
while not (PROJECT_ROOT / PROJECT_MARKER).is_file() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / PROJECT_MARKER).is_file():
    raise FileNotFoundError("Open this notebook from the project directory or one of its subdirectories.")
import sys
SRC_DIR = str(PROJECT_ROOT / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import json
import pandas as pd

train_path = PROJECT_ROOT / "data" / "processed" / "samsum-train-processed.csv"
if not train_path.is_file():
    raise FileNotFoundError("Processed training CSV is missing; follow the dataset setup guide first.")
train = pd.read_csv(train_path, dtype={"id": str})
sample_id = "13818513"
matches = train.loc[train["id"] == sample_id, "dialogue"]
if matches.empty:
    raise ValueError(f"Training sample {sample_id} was not found.")
dialogue = matches.iloc[0]
print(dialogue)

## Generate, validate, and call the tool

The helper applies the structured prompt, performs one corrective retry when needed, validates the final JSON against the project schema, and flags fallback output for human review. Only schema-valid results are sent to the allowlisted validation function.

In [ ]:
from structured_output.run_structured_output import MODEL_NAME, generate_structured, load_model
from integrated_demo.presentation import format_final_response_for_review, format_summary_for_review
from tool_calling.run_tool_calling import TOOL_NAME, execute_tool_call, generate_final_response

tokenizer, model = load_model()
generation = generate_structured(dialogue, tokenizer, model)
coverage_flags = generation["commitment_coverage_flags"]
print(f"Model: {MODEL_NAME}")
print(
    format_summary_for_review(
        generation["final_object"],
        schema_valid=generation["schema_valid"],
        used_fallback=generation["used_fallback"],
        validation_errors=generation["validation_errors"],
        coverage_flags=coverage_flags,
    )
)
print(
    "Generation diagnostics: "
    f"parse_ok={generation['parse_ok']}, "
    f"repair_attempted={generation['repair_attempted']}, "
    f"recovered={generation['recovered']}"
)

In [ ]:
if not generation["schema_valid"] or generation["used_fallback"]:
    raise RuntimeError(f"No schema-valid object to pass to the tool: {generation['validation_errors']}")
tool_request = {
    "tool_name": TOOL_NAME,
    "arguments": {"output": generation["final_object"]},
}
tool_result = execute_tool_call(tool_request)
tool_status = "passed" if tool_result["valid"] else "failed"
print(f"Tool validation: {tool_status}")
if not tool_result["valid"]:
    final_response = None
    final_response_status = "blocked_tool_validation_failed"
else:
    final_response = generate_final_response(
        dialogue,
        json.dumps(tool_request, ensure_ascii=False),
        tool_result,
        tokenizer,
        model,
    )
    if coverage_flags:
        final_response += "\n\nCompleteness review required: " + " ".join(coverage_flags)
    final_response_status = (
        "generated_after_valid_tool_result_with_coverage_warning"
        if coverage_flags
        else "generated_after_valid_tool_result"
    )
print("Final-response status:", final_response_status)
if final_response is not None:
    print()
    print(format_final_response_for_review(final_response))
print("\nHuman review is required; schema validation does not establish factual correctness.")

## Limitations

The function validates output structure only. A narrow first-person-commitment heuristic flags some likely omissions, but it does not repair output or prove factual support/completeness. Human review remains mandatory. The demonstration uses one training example for workflow illustration, not evaluation. The tool call is an application-level JSON protocol rather than native model-specific tool calling.